# StageOps: integración de las seis fuentes

**Hackatón SO-2026-041 · Investigación de incidencias en la venta de entradas**

## Contexto
Los notebooks 01-06 limpiaron y verificaron cada CSV por separado y dejaron en `docs/` una guía por fichero con lo que se había **demostrado**, lo que quedaba **pendiente** y cómo **unir** cada tabla con las demás. Este notebook cierra ese trabajo: integra las seis fuentes siguiendo las relaciones del anexo técnico, aplica las reglas del encargo y responde a las cinco preguntas de dirección.

`Documentación (docs/*.md) → Contrato de relaciones → Tabla maestra por reserva → Estado de cada reserva → Repeticiones legítimas vs conflictos → Mecanismo → Alcance → Validación → Exportación`

Todo se calcula en este notebook a partir de `data/clean/` (a su vez generado desde `data/raw/`, que no se modifica). Ninguna cifra está escrita a mano: si cambian los datos, cambian los resultados.

## 1. Método de trabajo

Cada bloque responde a una pregunta del encargo y se apoya en lo que dicen los documentos de cada fichero:

| Pregunta de dirección | Dónde se resuelve |
| :--- | :--- |
| 1. ¿Qué conciertos, periodos y compañías presentan incidencias demostrables? | apartados 6, 8 y 9 |
| 2. ¿Qué parece repetido pero es legítimo y qué es un conflicto real? | apartado 7 |
| 3. ¿Qué secuencia explica los casos afectados? | apartados 8 y 9 |
| 4. ¿Cuántas butacas, reservas, compradores e importes están afectados? | apartado 10 |
| 5. ¿Qué corregir y cómo comprobarlo? | apartado 14 |

Principios que se mantienen en todo el notebook:

1. **Elegir la unidad de análisis antes de unir** y comprobar claves y cardinalidades con `validate=` en cada `merge`.
2. **Un reloj por pregunta**: la regla de emisión depende de cuándo **llega** la confirmación (`capture_received_at`), no de cuándo se cobró.
3. **Demostrado no es lo mismo que explicado**: una coincidencia de fechas con un cambio de configuración no prueba causalidad; se contrasta con un grupo de comparación.
4. **Cada indicador declara su denominador**, para no duplicar personas ni dinero.

## 2. Importaciones y ubicación de los archivos

La raíz del repositorio se localiza subiendo desde la carpeta de trabajo hasta encontrar `data/clean`. Las constantes del servicio (duración de la reserva, cierre de datos) salen del encargo y se definen una sola vez.

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CARPETA_BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'clean').is_dir())
CARPETA_LIMPIOS = CARPETA_BASE / 'data' / 'clean'
CARPETA_DOCS = CARPETA_BASE / 'docs'
CARPETA_SALIDAS = CARPETA_BASE / 'outputs'
CARPETA_SALIDAS.mkdir(exist_ok=True)

SNAPSHOT_AT = pd.Timestamp('2026-10-28 23:59:59', tz='UTC')   # cierre de datos del encargo
FORMATO_FECHA = '%Y-%m-%dT%H:%M:%SZ'                          # ISO 8601 UTC

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 110)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print('Carpeta base:', CARPETA_BASE)
print('Ficheros limpios disponibles:', sorted(p.name for p in CARPETA_LIMPIOS.glob('*.csv')))

## 3. Lo que dicen los documentos: pendientes heredados

Cada notebook dejó en `docs/<fichero>_conclusiones.md` una lista de **pendientes para la integración**. En lugar de copiarlos a mano, se leen de los propios documentos: así el notebook de integración parte exactamente de lo que cada responsable dejó abierto, y si alguien añade un pendiente nuevo aparece aquí sin tocar el código.

In [ ]:
patron_checklist = re.compile(r'^\s*[-*]\s*\[ \]\s*(.+?)\s*$', re.M)
patron_problema = re.compile(r'^\|\s*(P\d)\s*\|\s*(.+?)\s*\|\s*(.+?)\s*\|\s*(.+?)\s*\|\s*$', re.M)

filas_pendientes, filas_problemas = [], []
for ruta in sorted(CARPETA_DOCS.glob('*conclusiones.md')):
    texto = ruta.read_text(encoding='utf-8')
    filas_pendientes += [{'documento': ruta.name, 'pendiente': item} for item in patron_checklist.findall(texto)]
    filas_problemas += [{'documento': ruta.name, 'id': i, 'problema': p, 'como_se_confirmaria': c, 'reservation_state': e.replace('**', '')}
                        for i, p, c, e in patron_problema.findall(texto)]

df_pendientes = pd.DataFrame(filas_pendientes)
df_problemas = pd.DataFrame(filas_problemas)

print(f'Documentos leídos: {len(list(CARPETA_DOCS.glob("*conclusiones.md")))} | pendientes en lista: {len(df_pendientes)} | problemas P1-Pn: {len(df_problemas)}\n')
display(df_problemas[['id', 'problema', 'reservation_state']])
display(df_pendientes)

**Qué hay que cerrar.** Los pendientes se agrupan en cuatro familias, y el notebook las resuelve en este orden:

| Familia | Pendientes que cierra | Apartado |
| :--- | :--- | :--- |
| Cobertura y claves entre ficheros | cada `reservation_id`, `payment_id` y `concert_id` existe donde debe; cuántas reservas no tienen pago | 5 |
| Confirmación frente a caducidad | P1 y las comprobaciones de «dentro/fuera de `[created_at, expires_at)`»; devoluciones rápidas frente a tardías | 6 |
| Localidades y entradas | P2 (localidad ocupada), doble vigencia, reventas legítimas tras anulación | 7 |
| Mecanismo y comparación | P6 y `CHG-004`: antes/después y frente a LumaTicket | 8 y 9 |
| Dinero | P3, P4, P5: capturas sin entrada, devoluciones, reenvíos | 6 y 10 |

## 4. Carga y normalización de las tablas limpias

Las seis tablas salen de notebooks distintos y, al compararlas, **no son idénticas en formato ni en vocabulario**. Se documenta aquí qué se encontró y cómo se neutraliza, **sin modificar ningún CSV** (cada autor regenera el suyo): la normalización ocurre al cargar, en una sola función, de modo que el resto del notebook trabaja con un esquema común.

| Diferencia encontrada | Efecto si no se trata | Cómo se neutraliza |
| :--- | :--- | :--- |
| **BOM**: lo conservan `customers` y `tickets`, no el resto | el BOM se pegaría al nombre de la primera columna | se lee siempre con `utf-8-sig` (funciona con y sin BOM) |
| **Formato de fecha**: `reservations` usa `2026-10-16 10:00:10+00:00` y el resto `2026-10-16T10:00:10Z` | comparar o ordenar como texto daría resultados distintos | todas las fechas se interpretan y pasan a `datetime64[ns, UTC]` |
| **Resolución temporal**: según la lectura, `s`, `us` o `ns` | `merge_asof` falla al comparar tipos distintos | se unifica a nanosegundos |
| **Vocabulario de pagos**: la misma idea se llama `occurred_at`, `captured_at` o `capture_occurred_at` según el fichero | uniones y filtros con nombres distintos para el mismo concepto | se adopta `<evento>_<campo>` en la tabla por reserva |

In [ ]:
BOMS = {}                  # qué ficheros traían BOM
FORMATOS_ORIGINALES = []   # formato de fecha de cada columna ANTES de normalizar

# Los ficheros de pagos nombran el mismo concepto de tres maneras. Se adopta <evento>_<campo>:
RENOMBRADO_COLUMNAS = {
    'payment_by_reservation': {'captured_at': 'capture_occurred_at', 'captured_eur': 'capture_amount_eur',
                               'refunded_at': 'refund_occurred_at', 'refunded_eur': 'refund_amount_eur'},
}


def leer_limpio(nombre, fechas=()):
    '''Se lee un CSV de data/clean y se normaliza: sin BOM, fechas en UTC con una sola resolución y nombres comunes.'''
    ruta = CARPETA_LIMPIOS / f'{nombre}.csv'
    BOMS[nombre] = ruta.read_bytes()[:3] == b'\xef\xbb\xbf'
    df = pd.read_csv(ruta, encoding='utf-8-sig')
    for columna in fechas:
        formato = df[columna].dropna().astype(str).str.replace(r'\d', '9', regex=True).unique()
        FORMATOS_ORIGINALES.append({'tabla': nombre, 'columna': columna, 'formato': formato[0]})
        df[columna] = pd.to_datetime(df[columna], utc=True).astype('datetime64[ns, UTC]')
    return df.rename(columns=RENOMBRADO_COLUMNAS.get(nombre, {}))


df_concerts = leer_limpio('concerts_clean', ['sales_open_at', 'doors_open_at', 'concert_start_at', 'snapshot_at'])
df_customers = leer_limpio('customers_clean')
df_reservations = leer_limpio('reservations_clean', ['created_at', 'expires_at'])
df_pagos = leer_limpio('payment_by_reservation', ['captured_at', 'capture_received_at', 'capture_last_received_at', 'refunded_at', 'refund_received_at'])
df_tickets = leer_limpio('tickets_clean', ['issued_at', 'voided_at'])
df_cambios = leer_limpio('system_changes_clean', ['effective_at', 'valid_from', 'valid_to'])

resumen = pd.DataFrame({
    'tabla': ['concerts', 'customers', 'reservations', 'payment_by_reservation', 'tickets', 'system_changes'],
    'unidad (una fila por...)': ['concierto', 'comprador', 'reserva', 'reserva con pago', 'entrada', 'cambio de configuración'],
    'filas': [len(df_concerts), len(df_customers), len(df_reservations), len(df_pagos), len(df_tickets), len(df_cambios)],
}).set_index('tabla')
display(resumen)

print('Lo que había en los ficheros ANTES de normalizar:')
formatos = (pd.DataFrame(FORMATOS_ORIGINALES).groupby('tabla')['formato']
            .agg(lambda s: ' · '.join(sorted(set(s)))).rename('formato de fecha en el CSV'))
display(pd.concat([formatos, pd.Series(BOMS, name='con BOM')], axis=1).fillna({'formato de fecha en el CSV': '(sin columnas de fecha)'}).rename_axis('tabla'))

**Lo que se encontró:** el formato de fecha de `reservations` es el único distinto (separador de espacio y desplazamiento `+00:00`, en lugar de `T` y `Z`), y el BOM aparece solo en `customers` y `tickets`. Tras cargar, todas las fechas son `datetime64[ns, UTC]` y las diferencias de formato dejan de importar.

### 4.1 Convención de nombres de columna

El esquema común que se aplica y se **audita** a continuación:

| Patrón | Significado | Ejemplos |
| :--- | :--- | :--- |
| `*_id` | identificador, siempre texto | `reservation_id`, `payment_id`, `concert_id` |
| `*_at` | instante en UTC (`datetime64[ns, UTC]`) | `created_at`, `capture_received_at` |
| `*_eur` | importe en euros | `unit_price_eur`, `capture_amount_eur` |
| `<evento>_<campo>` | campo de un evento de pago | `capture_occurred_at`, `refund_amount_eur` |
| `*_n_*`, `n_*` | recuentos | `capture_n_deliveries` |
| `is_*`, `has_*` | indicadores verdadero/falso | `is_late`, `has_ticket` |
| `*_s` | segundos | `confirmation_lag_s` |

Las **tablas de datos** (las que se unen y se exportan para análisis) usan columnas en inglés y `snake_case`. Las **tablas de presentación** para dirección (indicadores de alcance, importes) usan encabezados en español, porque las lee una persona.

In [ ]:
TABLAS = {'concerts': df_concerts, 'customers': df_customers, 'reservations': df_reservations,
          'payment_by_reservation': df_pagos, 'tickets': df_tickets, 'system_changes': df_cambios}
FECHAS_SIN_SUFIJO_AT = {'valid_from', 'valid_to'}   # intervalos de vigencia de system_changes
CLAVES = ['reservation_id', 'order_id', 'payment_id', 'concert_id', 'customer_id', 'seat_id', 'ticketing_company_id']

incumplimientos = []
for nombre, tabla in TABLAS.items():
    for columna in tabla.columns:
        es_fecha = isinstance(tabla[columna].dtype, pd.DatetimeTZDtype)
        if not re.fullmatch(r'[a-z][a-z0-9_]*', columna):
            incumplimientos.append((nombre, columna, 'no está en snake_case minúsculas'))
        if es_fecha and str(tabla[columna].dtype) != 'datetime64[ns, UTC]':
            incumplimientos.append((nombre, columna, f'fecha que no es UTC con resolución ns ({tabla[columna].dtype})'))
        if es_fecha and not (columna.endswith('_at') or columna in FECHAS_SIN_SUFIJO_AT):
            incumplimientos.append((nombre, columna, 'fecha sin sufijo _at'))
        if 'eur' in columna.split('_') and not columna.endswith('_eur'):
            incumplimientos.append((nombre, columna, 'importe sin sufijo _eur'))
        if columna in CLAVES and not pd.api.types.is_string_dtype(tabla[columna]):
            incumplimientos.append((nombre, columna, 'identificador que no es texto'))

assert not incumplimientos, f'el esquema no es uniforme: {incumplimientos}'
print('Esquema uniforme en las seis tablas: snake_case, fechas UTC (ns) con sufijo _at, importes con sufijo _eur e identificadores como texto.\n')

presencia = {c: ', '.join(n for n, t in TABLAS.items() if c in t.columns) for c in CLAVES}
display(pd.Series(presencia, name='tablas donde aparece').rename_axis('clave'))

## 5. Contrato de relaciones y cobertura entre ficheros

El anexo técnico define el modelo de información: el punto de enlace es `reservation_id`; `concerts` y `customers` son catálogos 1:N de las reservas; `tickets` es 1:0..1 respecto a las reservas; y cada entrada se respalda en una captura (`payment_id`). Se traduce ese modelo a una lista de relaciones y se comprueba cada una **con `validate=` de pandas**, de modo que el notebook falle si una clave deja de ser única en vez de multiplicar filas en silencio.

Una relación se lee «cada fila de **hijo** apunta a una de **padre**»: se mide cuántas filas hijo no encuentran padre (huérfanas) y si la unión conserva el número de filas.

In [ ]:
RELACIONES = [
    # (hijo, padre, clave, cardinalidad esperada, tabla hijo, tabla padre)
    ('reservations', 'concerts', 'concert_id', 'many_to_one', df_reservations, df_concerts),
    ('reservations', 'customers', 'customer_id', 'many_to_one', df_reservations, df_customers),
    ('payment_by_reservation', 'reservations', 'reservation_id', 'one_to_one', df_pagos, df_reservations),
    ('tickets', 'reservations', 'reservation_id', 'one_to_one', df_tickets, df_reservations),
    ('tickets', 'concerts', 'concert_id', 'many_to_one', df_tickets, df_concerts),
    ('tickets', 'payment_by_reservation', 'payment_id', 'one_to_one', df_tickets, df_pagos),
]

filas = []
for hijo, padre, clave, cardinalidad, t_hijo, t_padre in RELACIONES:
    union = t_hijo[[clave]].merge(t_padre[[clave]], on=clave, how='left', validate=cardinalidad, indicator=True)
    filas.append({'hijo': hijo, 'padre': padre, 'clave': clave, 'cardinalidad': cardinalidad,
                  'filas hijo': len(t_hijo), 'filas tras unir': len(union),
                  'huérfanas (hijo sin padre)': int((union['_merge'] == 'left_only').sum())})
df_relaciones = pd.DataFrame(filas)
display(df_relaciones)

assert (df_relaciones['filas hijo'] == df_relaciones['filas tras unir']).all(), 'alguna unión multiplica o pierde filas'
assert (df_relaciones['huérfanas (hijo sin padre)'] == 0).all(), 'hay registros huérfanos'
print('Todas las relaciones cumplen su cardinalidad y no hay registros huérfanos.')

In [ ]:
# Dirección contraria: padres sin hijos. Aquí sí se espera que haya, y lo que importa es cuántos y por qué.
reservas_sin_pago = ~df_reservations['reservation_id'].isin(df_pagos['reservation_id'])
reservas_sin_entrada = ~df_reservations['reservation_id'].isin(df_tickets['reservation_id'])
pagos_sin_entrada = ~df_pagos['reservation_id'].isin(df_tickets['reservation_id'])

print(f'Reservas sin pago:                      {int(reservas_sin_pago.sum()):>7,} de {len(df_reservations):,}')
print(f'Reservas sin entrada:                   {int(reservas_sin_entrada.sum()):>7,}')
print(f'Pagos (capturas) sin entrada emitida:   {int(pagos_sin_entrada.sum()):>7,} de {len(df_pagos):,}')

# Coherencia de los campos que se repiten en dos ficheros para la misma reserva
cruce_tickets = df_tickets.merge(df_reservations[['reservation_id', 'concert_id', 'seat_id']], on='reservation_id',
                                 suffixes=('', '_reserva'), validate='one_to_one')
cruce_pagos = df_tickets.merge(df_pagos[['reservation_id', 'payment_id']], on='reservation_id',
                               suffixes=('', '_pago'), validate='one_to_one')
print('\nEntradas cuyo concert_id difiere del de su reserva:', int((cruce_tickets['concert_id'] != cruce_tickets['concert_id_reserva']).sum()))
print('Entradas cuyo seat_id difiere del de su reserva:   ', int((cruce_tickets['seat_id'] != cruce_tickets['seat_id_reserva']).sum()))
print('Entradas cuyo payment_id difiere del de su pago:   ', int((cruce_pagos['payment_id'] != cruce_pagos['payment_id_pago']).sum()))

**Resultado de la cobertura:** las relaciones son todas coherentes: ninguna clave huérfana y ninguna unión cambia el número de filas. Los campos repetidos entre ficheros (`concert_id`, `seat_id`, `payment_id`) coinciden en todas las reservas con entrada.

Las diferencias de tamaño entre tablas **no son errores**, son el flujo normal de una compra:
- Las reservas sin pago son reservas que **caducaron sin confirmación**: el bloqueo de diez minutos terminó y la butaca quedó libre. Es lo que el encargo describe y no es una incidencia.
- Los pagos sin entrada son capturas a las que **no se emitió entrada**. Que no haya entrada no implica error: si la confirmación llegó con la reserva caducada, la regla dice que se devuelva el importe sin emitir. Hay que ver si, en efecto, se devolvió (apartado 6).

## 6. Tabla maestra por reserva y estado de cada reserva

### 6.1 Tabla maestra

La unidad de análisis es la **reserva**: es el punto de enlace del modelo y la única unidad a la que se pueden unir pagos y entradas sin multiplicar filas. Se construye con una unión por tabla, cada una con su `validate`:

| Unión | Cardinalidad | Por qué |
| :--- | :--- | :--- |
| reservations → concerts | `many_to_one` | aporta compañía y apertura de ventas |
| → payment_by_reservation | `one_to_one` | una captura y como máximo una devolución por reserva, ya resumidas en una fila |
| → tickets | `one_to_one` | una reserva tiene cero o una entrada |

Los pagos se unen desde `payment_by_reservation` y **no** desde las entregas: unir las entregas multiplicaría las reservas con reenvío (aviso del anexo técnico).

In [ ]:
df_maestra = (df_reservations
              .merge(df_concerts[['concert_id', 'ticketing_company_id', 'ticketing_company_name', 'sales_open_at']],
                     on='concert_id', how='left', validate='many_to_one')
              .merge(df_pagos, on='reservation_id', how='left', validate='one_to_one')
              .merge(df_tickets.drop(columns=['payment_id', 'concert_id', 'seat_id']),
                     on='reservation_id', how='left', validate='one_to_one'))

assert len(df_maestra) == len(df_reservations), 'la tabla maestra no tiene una fila por reserva'
print(f'df_maestra: {len(df_maestra):,} filas (una por reserva) y {df_maestra.shape[1]} columnas')

df_maestra['has_payment'] = df_maestra['capture_amount_eur'].notna()
df_maestra['has_ticket'] = df_maestra['ticket_id'].notna()
df_maestra['is_refunded'] = df_maestra['has_refund'].eq(True)

display(df_maestra[['reservation_id', 'ticketing_company_id', 'concert_id', 'seat_id', 'created_at', 'expires_at',
                    'capture_received_at', 'capture_amount_eur', 'refund_amount_eur', 'ticket_id', 'issued_at', 'voided_at', 'issuer_version']].head())

### 6.2 Qué reloj decide si una confirmación llega a tiempo

El encargo fija que solo puede emitirse entrada si **al recibirse la confirmación** la reserva sigue vigente, es decir, si la recepción cae dentro de `[created_at, expires_at)`. La guía de `payment_events` recuerda que hay dos recepciones posibles: la **primera entrega** (`capture_received_at`) y el **reenvío** (`capture_last_received_at`). Antes de elegir, se comprueba cuál de las dos gobierna la emisión: si la entrada se emite siempre a un intervalo fijo de una de ellas, esa es la que cuenta.

In [ ]:
con_entrada = df_maestra[df_maestra['has_ticket']]
desde_primera = (con_entrada['issued_at'] - con_entrada['capture_received_at']).dt.total_seconds()
desde_ultima = (con_entrada['issued_at'] - con_entrada['capture_last_received_at']).dt.total_seconds()

print('Segundos entre la PRIMERA recepción y la emisión:', desde_primera.value_counts().head(5).to_dict())
print('Segundos entre la ÚLTIMA recepción y la emisión: ', desde_ultima.describe().round(1)[['min', '50%', 'max']].to_dict())

reenvios_cruzan_plazo = df_maestra[df_maestra['has_payment']
                                   & (df_maestra['capture_received_at'] < df_maestra['expires_at'])
                                   & (df_maestra['capture_last_received_at'] >= df_maestra['expires_at'])]
print(f'\nReservas cuya primera entrega llega a tiempo pero cuyo reenvío cruza la caducidad: {len(reenvios_cruzan_plazo):,}')

**Resultado:** la entrada se emite siempre **exactamente 2 segundos después de la primera recepción** de la captura, sin una sola excepción, y no guarda relación con el reenvío. Por tanto, el reloj de la regla es `capture_received_at` (primera entrega). Los reenvíos no cambian ninguna decisión: no hay ninguna reserva cuya primera entrega llegue a tiempo y cuyo reenvío cruce la caducidad.

### 6.3 Estado de cada reserva

`reservations` no trae un estado final precalculado: «debe interpretarse con el plazo, los pagos y las entradas». Se deriva aplicando las reglas del servicio, de modo que cada reserva caiga en **un único estado**:

| Estado | Condición | ¿Es correcto según las reglas? |
| :--- | :--- | :--- |
| `SIN_PAGO_CADUCADA` | no hay captura | Sí: la butaca quedó libre |
| `CONFIRMADA_A_TIEMPO_CON_ENTRADA` | confirmación recibida antes de `expires_at` y entrada vigente | Sí |
| `CONFIRMADA_A_TIEMPO_ANULADA_Y_DEVUELTA` | a tiempo, entrada anulada y dinero devuelto | Sí: anulación con devolución |
| `TARDIA_DEVUELTA_SIN_ENTRADA` | confirmación recibida en o después de `expires_at`, sin entrada y con devolución | **Sí**: es exactamente lo que ordena la regla |
| `TARDIA_CON_ENTRADA` | confirmación recibida en o después de `expires_at` y entrada emitida | **No**: incumple la regla de emisión |

El instante de caducidad queda **excluido** de la vigencia, de ahí que `capture_received_at >= expires_at` sea «tarde». Si alguna reserva no encajara en ningún estado, el notebook se detiene.

In [ ]:
tarde = df_maestra['has_payment'] & (df_maestra['capture_received_at'] >= df_maestra['expires_at'])
a_tiempo = df_maestra['has_payment'] & ~tarde
vigente = df_maestra['voided_at'].isna()

df_maestra['reservation_state'] = np.select(
    [~df_maestra['has_payment'],
     a_tiempo & df_maestra['has_ticket'] & vigente,
     a_tiempo & df_maestra['has_ticket'] & ~vigente & df_maestra['is_refunded'],
     tarde & ~df_maestra['has_ticket'] & df_maestra['is_refunded'],
     tarde & df_maestra['has_ticket']],
    ['SIN_PAGO_CADUCADA', 'CONFIRMADA_A_TIEMPO_CON_ENTRADA', 'CONFIRMADA_A_TIEMPO_ANULADA_Y_DEVUELTA',
     'TARDIA_DEVUELTA_SIN_ENTRADA', 'TARDIA_CON_ENTRADA'],
    default='SIN_CLASIFICAR')
df_maestra['is_late'] = tarde

assert (df_maestra['reservation_state'] != 'SIN_CLASIFICAR').all(), 'hay reservas que no encajan en ningún estado'

tabla_estados = (df_maestra.groupby(['reservation_state', 'ticketing_company_id']).size().unstack(fill_value=0)
                 .assign(total=lambda t: t.sum(axis=1)))
tabla_estados.loc['TOTAL'] = tabla_estados.sum()
display(tabla_estados)

print('Dinero cobrado por estado (EUR):')
display(df_maestra.groupby('reservation_state')[['capture_amount_eur', 'refund_amount_eur']].sum())

**Lo que muestran los estados:** las 188.788 reservas encajan en cinco estados sin dejar ninguna sin clasificar. Cuatro de ellos son comportamientos **correctos**: reservas que caducaron sin pago, ventas confirmadas a tiempo, anulaciones con devolución y confirmaciones tardías que se devolvieron sin emitir entrada.

El quinto, **`TARDIA_CON_ENTRADA`**, es el único que incumple las reglas del servicio: se recibió la confirmación cuando la reserva ya había caducado y, en lugar de devolver el importe, **se emitió la entrada** y no se devolvió nada. Y aparece **solo en EntradaLink**. LumaTicket tiene confirmaciones tardías (por eso hay devoluciones sin entrada en ambas compañías) pero las gestiona siempre como manda la regla.

## 7. Repeticiones legítimas frente a conflictos reales

El encargo pide distinguir tres situaciones que **parecen** repeticiones: reenvíos de notificaciones, nuevas reservas tras una caducidad y ventas tras una anulación, de lo que sí incumple las reglas: **dos entradas simultáneamente vigentes para la misma localidad**. Se comprueba cada una por separado. Una localidad es la pareja `concert_id + seat_id`; el mismo `seat_id` en conciertos distintos no cuenta.

### 7.1 Reenvíos de notificaciones (legítimos)

In [ ]:
reservas_con_reenvio = df_pagos[df_pagos['capture_n_deliveries'] > 1]
print(f'Reservas cuya captura se entregó más de una vez: {len(reservas_con_reenvio):,}')
print(f'Capturas contadas una sola vez en la tabla de pagos: {len(df_pagos):,} (una por reserva)')
print(f'Dinero cobrado contando cada captura una vez: {df_pagos["capture_amount_eur"].sum():,.0f} EUR')
print(f'  del que corresponde a reservas con reenvío:  {reservas_con_reenvio["capture_amount_eur"].sum():,.0f} EUR (no se cobra dos veces)')

Los reenvíos **no son nuevos cobros**: cada reserva con reenvío se cuenta una vez, y el dinero cobrado se calcula sobre capturas únicas (en `payment_events` sumar todas las filas habría inflado el cobrado). Son legítimos.

### 7.2 Nuevas reservas de una localidad tras una caducidad (legítimas)

Una localidad puede tener varias reservas **en momentos distintos**. Lo que sería anómalo es que dos reservas la bloquearan **a la vez**. Como todas las reservas duran lo mismo (diez minutos), basta comparar cada reserva con la anterior de la misma localidad: hay solape si empieza antes de que la anterior termine.

In [ ]:
reservas_ordenadas = df_reservations.sort_values(['concert_id', 'seat_id', 'created_at']).copy()
reservas_ordenadas['caducidad_anterior'] = reservas_ordenadas.groupby(['concert_id', 'seat_id'])['expires_at'].shift()
reservas_ordenadas['solapa'] = reservas_ordenadas['created_at'] < reservas_ordenadas['caducidad_anterior']

por_localidad = reservas_ordenadas.groupby(['concert_id', 'seat_id']).size()
print(f'Localidades con al menos una reserva: {len(por_localidad):,}')
print(f'Localidades reservadas más de una vez: {int((por_localidad > 1).sum()):,}')
print(f'Reservas que bloquean una localidad ya bloqueada por otra (solape de ventanas): {int(reservas_ordenadas["solapa"].sum()):,}')

**Resultado:** muchas localidades se reservan varias veces, pero **ninguna reserva se solapa con otra** de la misma localidad. Cada nueva reserva llega cuando la anterior ya había caducado. Es el comportamiento legítimo que describe el encargo, y la capa de reservas funciona bien. El problema, si existe, no está en el bloqueo.

### 7.3 Ventas tras una anulación (legítimas) y dobles entradas vigentes (conflicto)

Ahora a nivel de **entradas**. Cada entrada está vigente en `[issued_at, voided_at)`, o hasta el cierre si no se anuló. Dos entradas sucesivas de la misma localidad son legítimas (la primera se anuló antes de emitir la segunda); dos entradas cuyos intervalos se solapan, no.

In [ ]:
entradas = df_tickets.copy()
entradas['fin'] = entradas['voided_at'].fillna(SNAPSHOT_AT)
entradas = entradas.sort_values(['concert_id', 'seat_id', 'issued_at']).reset_index(drop=True)

# Fin máximo de todas las entradas anteriores de la misma localidad (se trabaja en naïve UTC para poder usar cummax)
fin_naive = entradas['fin'].dt.tz_convert(None)
entradas['fin_max_anterior'] = fin_naive.groupby([entradas['concert_id'], entradas['seat_id']]).cummax()
entradas['fin_max_anterior'] = entradas.groupby(['concert_id', 'seat_id'])['fin_max_anterior'].shift()
entradas['tiene_anterior'] = entradas['fin_max_anterior'].notna()
entradas['solapa'] = entradas['issued_at'].dt.tz_convert(None) < entradas['fin_max_anterior']

n_entradas_por_localidad = entradas.groupby(['concert_id', 'seat_id']).size()
localidades_conflicto = entradas.loc[entradas['solapa'], ['concert_id', 'seat_id']].drop_duplicates()
localidades_reventa = (entradas.loc[entradas['tiene_anterior'] & ~entradas['solapa'], ['concert_id', 'seat_id']]
                       .drop_duplicates().merge(localidades_conflicto, how='left', indicator=True, on=['concert_id', 'seat_id']))
localidades_reventa = localidades_reventa[localidades_reventa['_merge'] == 'left_only']

print(f'Localidades con más de una entrada emitida:                   {int((n_entradas_por_localidad > 1).sum()):,}')
print(f'  · reventas legítimas (la anterior se anuló antes):          {len(localidades_reventa):,}')
print(f'  · CONFLICTO: dos entradas vigentes a la vez:                {len(localidades_conflicto):,}')

# Se marca cada entrada implicada en un conflicto, tanto la que solapa como aquella con la que solapa
claves_conflicto = set(zip(localidades_conflicto['concert_id'], localidades_conflicto['seat_id']))
entradas['en_conflicto'] = [(c, s) in claves_conflicto for c, s in zip(entradas['concert_id'], entradas['seat_id'])]
print(f'Entradas implicadas en conflictos: {int(entradas["en_conflicto"].sum()):,}')

display(localidades_conflicto.groupby('concert_id').size().to_frame('localidades en conflicto')
        .join(entradas[entradas['en_conflicto']].groupby('concert_id').agg(entradas=('ticket_id', 'size'), versiones=('issuer_version', lambda s: ', '.join(sorted(s.unique()))))))

**Resultado:** hay dos fenómenos que hay que mantener separados.

- Las **reventas tras anulación** son legítimas: en cada una, la entrada anterior se anuló antes de emitirse la siguiente.
- Los **conflictos reales** son localidades con dos entradas **vigentes a la vez**. Son exactamente las que ya había detectado el notebook de `tickets`, y aparecen **solo** en CON-04 y CON-06 (EntradaLink) y **solo** con la versión del emisor 4.8.0. En ningún concierto de LumaTicket ni en CON-02 hay un solo caso.

Falta saber **por qué** se emitieron dos entradas para una misma localidad. Eso exige mirar las reservas y los pagos que hay detrás, que es lo que sigue.

## 8. Mecanismo: de la confirmación tardía a la entrada duplicada

### 8.1 Las entradas emitidas tras caducidad

El estado `TARDIA_CON_ENTRADA` del apartado 6 identifica las reservas cuya confirmación llegó con el plazo vencido y que, aun así, emitieron entrada. Se cruzan con los conflictos de localidad del apartado 7 y con el tiempo que pasó entre la caducidad y la confirmación.

In [ ]:
afectadas = df_maestra[df_maestra['reservation_state'] == 'TARDIA_CON_ENTRADA'].copy()
afectadas['seconds_after_expiry'] = (afectadas['capture_received_at'] - afectadas['expires_at']).dt.total_seconds()
afectadas['seat_in_conflict'] = [(c, s) in claves_conflicto for c, s in zip(afectadas['concert_id'], afectadas['seat_id'])]

print(f'Reservas con entrada emitida tras caducidad: {len(afectadas):,}')
print('  Compañías:', afectadas['ticketing_company_id'].value_counts().to_dict())
print('  Conciertos:', afectadas['concert_id'].value_counts().sort_index().to_dict())
print('  Versiones del emisor:', afectadas['issuer_version'].value_counts().to_dict())
print('  Devueltas alguna vez:', int(afectadas['is_refunded'].sum()))
print('  Segundos entre la caducidad y la confirmación (mín / mediana / máx):',
      int(afectadas['seconds_after_expiry'].min()), '/', int(afectadas['seconds_after_expiry'].median()), '/', int(afectadas['seconds_after_expiry'].max()))
print('  Primera / última emisión:', afectadas['issued_at'].min(), '/', afectadas['issued_at'].max())

print('\nRelación con las localidades en conflicto:')
display(afectadas['seat_in_conflict'].map({True: 'localidad con dos entradas vigentes', False: 'localidad sin segunda entrada'})
        .value_counts().to_frame('reservas'))

**Lo que muestran los datos:** todas las entradas emitidas tras caducidad son de EntradaLink (CON-04 y CON-06), todas con la versión 4.8.0, ninguna se devolvió, y la confirmación llegó entre unos minutos y unos 8 minutos después de que la reserva caducara. Estas reservas se reparten en dos grupos:

- **Con localidad en conflicto:** mientras la confirmación tardaba, la reserva caducó, **la localidad quedó libre, se reservó y se vendió a otra persona**, y entonces llegó la confirmación tardía y se emitió también la entrada de la primera reserva. Resultado: dos entradas vigentes para la misma butaca.
- **Sin segunda entrada:** la localidad no se había vuelto a vender. No hay doble vigencia, pero la regla es explícita: *«El protocolo permite esa devolución aunque la butaca aún no se haya vendido de nuevo»*. La entrada tampoco debía emitirse. Es una incidencia de dinero y de regla, aunque no de butaca.

Por tanto **hay un único mecanismo, no dos**: la confirmación tardía emite entrada. Que acabe o no en un conflicto de butaca depende de si la localidad se revendió mientras tanto.

### 8.2 Contraste con la configuración vigente (`system_changes`)

La guía de `system_changes` advierte de que `CHG-004` cambia la regla `reservation_validation` de EntradaLink de `CURRENT_OWNER_AND_NOT_EXPIRED` a `PAYMENT_REFERENCE_EXISTS`, y de que **un cambio temporalmente próximo no demuestra por sí mismo la causa**. Por eso no basta con ver que las fechas coinciden: se comprueba **qué regla estaba vigente en el instante de cada confirmación** y se compara **cómo se trataron las confirmaciones tardías** con una y otra regla, y en LumaTicket, que no cambió la regla.

La consulta usa la misma `configuracion_vigente` validada en el notebook de `system_changes` (`merge_asof` por compañía, componente y parámetro; inicio incluido, fin excluido).

In [ ]:
def configuracion_vigente(cambios, instantes, componente, parametro, col_compania='ticketing_company_id', col_instante='instante'):
    '''Se devuelve, para cada fila de `instantes`, el valor de `parametro` vigente en ese instante (último cambio <= instante).'''
    historial = (cambios[(cambios['component'] == componente) & (cambios['setting_name'] == parametro)]
                 [['ticketing_company_id', 'effective_at', 'new_value', 'version', 'change_id']]
                 .rename(columns={'new_value': 'valor_vigente', 'version': 'version_vigente', 'change_id': 'change_id_vigente'})
                 .sort_values('effective_at'))
    consulta = instantes.copy()
    consulta['_orden'] = range(len(consulta))
    consulta[col_instante] = consulta[col_instante].astype('datetime64[ns, UTC]')
    consulta[col_compania] = consulta[col_compania].astype('string')
    historial['ticketing_company_id'] = historial['ticketing_company_id'].astype('string')
    union = pd.merge_asof(consulta.sort_values(col_instante), historial, left_on=col_instante, right_on='effective_at',
                          left_by=col_compania, right_by='ticketing_company_id', direction='backward', allow_exact_matches=True)
    return union.sort_values('_orden').drop(columns='_orden').reset_index(drop=True)


pagos_maestra = df_maestra[df_maestra['has_payment']].copy()
instantes = pagos_maestra[['reservation_id', 'ticketing_company_id']].assign(instante=pagos_maestra['capture_received_at'])
vigente_en_confirmacion = configuracion_vigente(df_cambios, instantes, 'ticket_issuer', 'reservation_validation')

assert len(vigente_en_confirmacion) == len(pagos_maestra), 'la consulta de configuración ha multiplicado o perdido filas'
pagos_maestra = pagos_maestra.merge(vigente_en_confirmacion[['reservation_id', 'valor_vigente', 'change_id_vigente']],
                                    on='reservation_id', how='left', validate='one_to_one')
pagos_maestra['resultado'] = np.where(pagos_maestra['has_ticket'], 'entrada emitida', 'devuelta sin entrada')

confirmaciones_tardias = pagos_maestra[pagos_maestra['is_late']]
tabla_regla = (confirmaciones_tardias.groupby(['ticketing_company_id', 'valor_vigente', 'resultado']).size()
               .unstack(fill_value=0).reindex(columns=['devuelta sin entrada', 'entrada emitida'], fill_value=0))
tabla_regla['% con entrada'] = tabla_regla['entrada emitida'] / tabla_regla.sum(axis=1) * 100
display(tabla_regla.rename_axis(['compañía', 'regla vigente en la confirmación']))

# ¿Cambió solo el tratamiento de las confirmaciones tardías, o también su frecuencia?
cambio_004 = df_cambios.loc[df_cambios['change_id'] == 'CHG-004', 'effective_at'].iloc[0]
pagos_maestra['periodo'] = np.where(pagos_maestra['capture_received_at'] < cambio_004, 'antes de CHG-004', 'desde CHG-004')
pagos_maestra['retraso_confirmacion_s'] = (pagos_maestra['capture_received_at'] - pagos_maestra['capture_occurred_at']).dt.total_seconds()

frecuencia = (pagos_maestra.groupby(['ticketing_company_id', 'concert_id', 'periodo'])
              .agg(confirmaciones=('reservation_id', 'size'), tardias=('is_late', 'sum'),
                   retraso_mediano_s=('retraso_confirmacion_s', 'median')))
frecuencia['% tardías'] = frecuencia['tardias'] / frecuencia['confirmaciones'] * 100
print('Frecuencia de confirmaciones tardías por concierto y periodo:')
display(frecuencia)

print('\nRetraso entre la captura en la pasarela y la recepción en ticketing (segundos):')
display(pagos_maestra.groupby(['ticketing_company_id', 'is_late'])['retraso_confirmacion_s']
        .agg(['count', 'min', 'median', 'max']).rename_axis(['compañía', 'confirmación tardía']))

**Lo que muestra el contraste:**

- **Con `CURRENT_OWNER_AND_NOT_EXPIRED` vigente, ninguna confirmación tardía emitió entrada**: ni en EntradaLink antes de `CHG-004` ni en LumaTicket durante todo el periodo. Todas se devolvieron sin emitir, como manda la regla.
- **Con `PAYMENT_REFERENCE_EXISTS` vigente (EntradaLink desde `CHG-004`), todas las confirmaciones tardías emitieron entrada**, sin una sola devolución.
- **Pero el tratamiento no es lo único que cambia: también cambia la frecuencia.** Las confirmaciones tardías son una población **aparte**: su retraso va de unos 516 a unos 1.063 segundos, mientras que ninguna confirmación normal supera los 355. Esa población existe en **todos** los conciertos y compañías con una frecuencia de base de alrededor del 1,1-1,3 %. En CON-04 (tras `CHG-004`) y en CON-06 sube a cerca del **4 %**, unas tres veces más. Antes de `CHG-004`, EntradaLink estaba en la línea de base (CON-02: 1,3 %; CON-04 en sus primeros 20 minutos: 0,8 %).
- LumaTicket funciona como **grupo de comparación**: mantiene su frecuencia de base (≈1,3 %) durante todo el periodo y trata siempre bien las tardías, también en CON-03 y CON-05, que abren las mismas fechas que CON-04 y CON-06.

**Qué se puede afirmar y qué no.**

1. **Demostrado:** con la regla antigua, toda confirmación tardía se devuelve sin emitir (EntradaLink antes de `CHG-004` y LumaTicket siempre); con la regla nueva, toda confirmación tardía emite entrada. Es la **asociación más fuerte** de los datos, y coincide con lo que el diccionario dice que hace `PAYMENT_REFERENCE_EXISTS`: no exige vigencia.
2. **Demostrado:** en EntradaLink desde `CHG-004` hay **más** confirmaciones tardías de lo habitual. Por tanto la causa del daño no es solo la regla: hacen falta **las dos cosas**, más confirmaciones lentas y una regla que ya no las filtra. Con la regla antigua, ese aumento habría producido más devoluciones y ningún conflicto.
3. **No demostrado:** por qué aumentan los retrasos. La descripción de `CHG-004` dice *«Actualización del adaptador de confirmación de pagos»*, lo que **encaja** con que la misma actualización afecte a la validación y a la latencia, pero los datos no lo prueban. Tampoco se puede descartar una causa ajena al cambio (carga en la apertura de ventas de esos dos conciertos, por ejemplo).

**No es una prueba de causalidad.** Para cerrarla haría falta ver el código o la configuración del emisor y del adaptador, los registros de latencia de la pasarela, o reproducir la regla anterior en un entorno de pruebas. Se deja como información adicional necesaria (apartado 14).

## 9. Un caso concreto, rastreable por identificador

Se toma el **primer caso en el tiempo** de una localidad con dos entradas vigentes cuya segunda entrada procede de una confirmación tardía, y se reconstruye su secuencia con los cinco relojes del modelo: bloqueo (`created_at`, `expires_at`), pasarela (`capture_occurred_at`), recepción en ticketing (`capture_received_at`) y emisión (`issued_at`).

In [ ]:
candidatas = afectadas[afectadas['seat_in_conflict']].sort_values('issued_at')
caso = candidatas.iloc[0]
localidad = df_maestra[(df_maestra['concert_id'] == caso['concert_id']) & (df_maestra['seat_id'] == caso['seat_id'])]
localidad = localidad[localidad['has_payment']].sort_values('created_at')

print(f'Localidad: {caso["concert_id"]} · {caso["seat_id"]} · compañía {caso["ticketing_company_id"]}\n')
display(localidad[['reservation_id', 'order_id', 'customer_id', 'payment_id', 'ticket_id', 'issuer_version', 'reservation_state']])

eventos = []
for _, f in localidad.iterrows():
    eventos += [(f['created_at'], 'Se crea la reserva y se bloquea la localidad', f['reservation_id']),
                (f['expires_at'], 'Caduca la reserva (la localidad queda libre)', f['reservation_id']),
                (f['capture_occurred_at'], 'La pasarela captura el cobro', f['reservation_id']),
                (f['capture_received_at'], 'Ticketing recibe la confirmación', f['reservation_id']),
                (f['issued_at'], 'Ticketing emite la entrada ' + str(f['ticket_id']), f['reservation_id'])]
df_secuencia = pd.DataFrame(eventos, columns=['at_utc', 'event', 'reservation']).sort_values('at_utc').reset_index(drop=True)
df_secuencia['seconds_from_start'] = (df_secuencia['at_utc'] - df_secuencia['at_utc'].iloc[0]).dt.total_seconds().astype(int)
display(df_secuencia)

tarde_caso = localidad[localidad['reservation_id'] == caso['reservation_id']].iloc[0]
print(f'La confirmación de {tarde_caso["reservation_id"]} llega {int((tarde_caso["capture_received_at"] - tarde_caso["expires_at"]).total_seconds())} s DESPUÉS de que su reserva caducara, '
      f'y se emite la entrada {tarde_caso["ticket_id"]} (versión {tarde_caso["issuer_version"]}) aunque la localidad ya tenía otra entrada vigente.')

**Lectura del caso:** la secuencia muestra el mecanismo completo. La primera reserva caduca; la localidad queda libre y **otra persona la reserva, la paga y recibe su entrada**; después llega la confirmación del primer cobro, ya fuera de plazo, y el emisor genera también una entrada para la primera reserva. Ninguna de las dos operaciones viola por sí sola el bloqueo de diez minutos (en 7.2 no hay solapes): el conflicto nace de **emitir una entrada con una confirmación fuera de plazo**.

## 10. Alcance: indicadores con su definición y su denominador

El encargo pide cuantificar «localidades, entradas, reservas o pedidos y compradores afectados», definiendo cada indicador para no duplicar personas ni dinero, y separar cobros, devoluciones e importes que **podrían requerir regularización**, sin presentarlos como pérdidas.

Se distinguen dos poblaciones:
- **Entrada indebida** (`TARDIA_CON_ENTRADA`): la reserva cuya confirmación tardía emitió entrada. Es la que incumple la regla.
- **Contraparte legítima**: en las localidades con doble vigencia, el otro titular, que reservó la localidad **después** de la caducidad y compró dentro de las reglas. No ha incumplido nada, pero **se ve perjudicado** por el conflicto.

In [ ]:
# La contraparte legítima: la otra entrada vigente de las localidades en conflicto
entradas_conflicto = entradas[entradas['en_conflicto']]
tickets_indebidos = set(afectadas['ticket_id'])
contraparte = df_maestra[df_maestra['ticket_id'].isin(set(entradas_conflicto['ticket_id']) - tickets_indebidos)]

pagos_total = int(df_maestra['has_payment'].sum())
pagos_el = int((df_maestra['has_payment'] & (df_maestra['ticketing_company_id'] == 'EL')).sum())
pagos_el_desde = int((pagos_maestra['periodo'].eq('desde CHG-004') & pagos_maestra['ticketing_company_id'].eq('EL')).sum())
compradores_indebidos = set(afectadas['customer_id'])
compradores_contraparte = set(contraparte['customer_id'])

indicadores = pd.DataFrame([
    ('Reservas con entrada indebida', 'reservas cuya confirmación llegó con la reserva caducada y emitieron entrada',
     len(afectadas), 'reservas con pago de EL desde CHG-004', pagos_el_desde),
    ('Entradas indebidas', 'una por reserva anterior (relación 1:0..1)',
     afectadas['ticket_id'].nunique(), 'entradas emitidas en EL', int(df_maestra['ticketing_company_id'].eq('EL').mul(df_maestra['has_ticket']).sum())),
    ('Localidades con dos entradas vigentes', 'pares concert_id + seat_id con intervalos de vigencia solapados',
     len(localidades_conflicto), 'localidades con alguna entrada', int(entradas.groupby(['concert_id', 'seat_id']).ngroups)),
    ('Entradas implicadas en esos conflictos', 'las dos de cada localidad en conflicto',
     int(entradas['en_conflicto'].sum()), 'entradas emitidas', len(df_tickets)),
    ('Entradas indebidas sin conflicto de butaca', 'entrada indebida cuya localidad no se volvió a vender',
     int((~afectadas['seat_in_conflict']).sum()), 'entradas indebidas', len(afectadas)),
    ('Pedidos con alguna entrada indebida', 'order_id distintos de las reservas indebidas',
     afectadas['order_id'].nunique(), 'pedidos en el fichero', df_maestra['order_id'].nunique()),
    ('Compradores con alguna entrada indebida', 'customer_id distintos de las reservas indebidas',
     len(compradores_indebidos), 'compradores con alguna reserva', df_maestra['customer_id'].nunique()),
    ('Compradores contraparte (perjudicados)', 'customer_id distintos de la otra entrada vigente de cada conflicto',
     len(compradores_contraparte), 'compradores con alguna reserva', df_maestra['customer_id'].nunique()),
    ('Compradores afectados en total (sin duplicar)', 'unión de ambos conjuntos',
     len(compradores_indebidos | compradores_contraparte), 'compradores con alguna reserva', df_maestra['customer_id'].nunique()),
], columns=['indicador', 'definición', 'valor', 'denominador', 'valor_denominador'])
indicadores['%'] = indicadores['valor'] / indicadores['valor_denominador'] * 100
display(indicadores.style.format({'valor': '{:,.0f}', 'valor_denominador': '{:,.0f}', '%': '{:.2f}'}))

In [ ]:
# Dinero: se separa lo cobrado, lo devuelto y lo que podría requerir regularización
dinero = pd.DataFrame([
    ('Cobrado en total (capturas únicas)', 'suma de capturas sin contar reenvíos', df_pagos['capture_amount_eur'].sum()),
    ('Devuelto en total', 'suma de devoluciones', df_pagos['refund_amount_eur'].sum()),
    ('  · devuelto correctamente por confirmación tardía', 'estado TARDIA_DEVUELTA_SIN_ENTRADA: la regla lo exige', df_maestra.loc[df_maestra['reservation_state'] == 'TARDIA_DEVUELTA_SIN_ENTRADA', 'refund_amount_eur'].sum()),
    ('  · devuelto por anulación de la entrada', 'estado CONFIRMADA_A_TIEMPO_ANULADA_Y_DEVUELTA', df_maestra.loc[df_maestra['reservation_state'] == 'CONFIRMADA_A_TIEMPO_ANULADA_Y_DEVUELTA', 'refund_amount_eur'].sum()),
    ('Cobrado en reservas con entrada indebida', 'importe que, según la regla, debió devolverse: PODRÍA REQUERIR REGULARIZACIÓN', afectadas['capture_amount_eur'].sum()),
    ('  · en localidades con doble vigencia', 'de ese importe, el de las entradas que además chocan con otra vigente', afectadas.loc[afectadas['seat_in_conflict'], 'capture_amount_eur'].sum()),
    ('  · en localidades sin segunda entrada', 'el resto', afectadas.loc[~afectadas['seat_in_conflict'], 'capture_amount_eur'].sum()),
    ('Cobrado en la contraparte legítima', 'importe de quienes compraron dentro de las reglas (no se regulariza)', contraparte['capture_amount_eur'].sum()),
], columns=['concepto', 'definición', 'EUR']).set_index('concepto')
display(dinero.style.format({'EUR': '{:,.0f}'}))

assert np.isclose(dinero.loc['Cobrado en total (capturas únicas)', 'EUR'], df_maestra['capture_amount_eur'].sum()), 'el cobrado de la maestra no coincide con el de los pagos'

print('Alcance por concierto:')
por_concierto = (df_maestra[df_maestra['has_payment']].groupby(['ticketing_company_id', 'concert_id'])
                 .agg(reservas_con_pago=('reservation_id', 'size'), tardias=('is_late', 'sum'))
                 .join(afectadas.groupby(['ticketing_company_id', 'concert_id']).agg(
                     entradas_indebidas=('ticket_id', 'size'), con_conflicto=('seat_in_conflict', 'sum'),
                     primera_emision=('issued_at', 'min'), ultima_emision=('issued_at', 'max'), eur=('capture_amount_eur', 'sum')), how='left')
                 .fillna({'entradas_indebidas': 0, 'con_conflicto': 0, 'eur': 0}))
por_concierto['% sobre reservas con pago'] = por_concierto['entradas_indebidas'] / por_concierto['reservas_con_pago'] * 100
display(por_concierto)

**Lectura del alcance:** el problema está **acotado**: afecta a dos de los seis conciertos, CON-04 y CON-06, ambos de EntradaLink, y solo desde que `CHG-004` entró en vigor. En esos conciertos afecta a una fracción pequeña pero real de las reservas con pago. El importe de las reservas con entrada indebida es la **cifra que podría requerir regularización**, no una pérdida: depende de si se devuelve a quien llegó tarde o se mantiene su entrada y se reubica a la contraparte, decisión que corresponde a StageOps (apartado 14).

Dos precisiones para no duplicar:
- Las **entradas** implicadas en conflictos son el doble que las **localidades**, pero solo la mitad de ellas (las tardías) son indebidas; la otra mitad es la contraparte legítima.
- Un **comprador** puede aparecer en varias reservas y un `customer_id` **no equivale a una persona** (los catálogos no identifican acompañantes), por eso los compradores se cuentan por `customer_id` distintos y no se suman a las reservas.

## 11. Gráficos para el informe (máximo dos)

**Gráfico 1.** Cómo se trató cada confirmación recibida con la reserva caducada, en tramos de 15 minutos, **el día en que entró en vigor `CHG-004`**. Se comparan EntradaLink (CON-04) y LumaTicket (CON-03), dos conciertos que abrieron ventas ese mismo día, con la misma exposición, y de los que solo uno cambió la regla. Es la evidencia central: la conducta de EntradaLink cambia de un tramo al siguiente en el momento del cambio, y LumaTicket no.

**Gráfico 2.** Alcance por concierto: entradas indebidas, separando las que acaban en doble vigencia de las que no.

In [ ]:
COLOR_CORRECTO, COLOR_INDEBIDO = '#2f6db5', '#d4452f'

# Conciertos que abrieron ventas el mismo día que entró en vigor CHG-004 (misma exposición para las dos compañías)
dia_cambio = cambio_004.floor('D')
mismo_dia = pagos_maestra[pagos_maestra['is_late'] & (pagos_maestra['sales_open_at'].dt.floor('D') == dia_cambio)].copy()
mismo_dia['tramo'] = mismo_dia['capture_received_at'].dt.floor('15min').dt.tz_convert(None)
tramos = mismo_dia.groupby(['concert_id', 'tramo', 'resultado']).size().unstack(fill_value=0).reindex(columns=['devuelta sin entrada', 'entrada emitida'], fill_value=0)
rejilla = pd.date_range(dia_cambio.tz_convert(None) + pd.Timedelta(hours=10), dia_cambio.tz_convert(None) + pd.Timedelta(hours=18, minutes=30), freq='15min')


def grafico_tratamiento(figsize, base, compacto=False):
    '''Gráfico 1: qué se hizo con cada confirmación tardía el día de CHG-004. `compacto` da la versión para el informe.'''
    fig, ejes = plt.subplots(2, 1, figsize=figsize, sharex=True, sharey=True)
    paneles = [('CON-04', 'EntradaLink · CON-04' if compacto else 'EntradaLink · CON-04 (cambia la regla a las 10:20 UTC)'),
               ('CON-03', 'LumaTicket · CON-03' if compacto else 'LumaTicket · CON-03 (misma apertura de ventas, sin cambio de regla)')]
    etiqueta_ok = 'Devuelta (correcto)' if compacto else 'Devuelta sin emitir entrada (correcto)'
    etiqueta_ko = 'Entrada emitida (incumple)' if compacto else 'Entrada emitida con la reserva caducada (incumple)'
    for eje, (concierto, titulo) in zip(ejes, paneles):
        datos = tramos.loc[concierto].reindex(rejilla, fill_value=0)
        ancho = 14 / 1440
        eje.bar(rejilla, datos['devuelta sin entrada'], width=ancho, align='edge', color=COLOR_CORRECTO, label=etiqueta_ok)
        eje.bar(rejilla, datos['entrada emitida'], bottom=datos['devuelta sin entrada'], width=ancho, align='edge', color=COLOR_INDEBIDO, label=etiqueta_ko)
        eje.axvline(cambio_004.tz_convert(None), color='black', linestyle='--', linewidth=1)
        eje.set_title(titulo, loc='left', fontsize=base + 1)
        eje.set_ylabel('tardías por\ntramo de 15 min' if compacto else 'confirmaciones tardías\npor tramo de 15 min', fontsize=base)
        eje.tick_params(labelsize=base)
    ejes[1].text(cambio_004.tz_convert(None) + pd.Timedelta(minutes=8), ejes[1].get_ylim()[1] * 0.97, 'CHG-004 (10:20)', ha='left', va='top', fontsize=base)
    ejes[1].legend(loc='upper right', fontsize=base, frameon=False)
    ejes[1].set_xlabel(f'Hora de recepción, {dia_cambio:%d/%m/%Y} (UTC)' if compacto else f'Hora de recepción de la confirmación, {dia_cambio:%d/%m/%Y} (UTC)', fontsize=base)
    ejes[1].xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%H:%M'))
    if not compacto:
        fig.suptitle('Confirmaciones recibidas con la reserva caducada: qué se hizo con cada una', fontsize=base + 1, x=0.01, ha='left')
    fig.tight_layout()
    return fig


fig = grafico_tratamiento((11, 6), 10)
fig.savefig(CARPETA_SALIDAS / 'grafico_1_tratamiento_confirmaciones_tardias.png', dpi=200, bbox_inches='tight')
plt.show()

# Versión compacta para el informe (misma función y mismos datos; solo cambian tamaño y rótulos)
fig_informe = grafico_tratamiento((3.9, 2.7), 6.5, compacto=True)
fig_informe.savefig(CARPETA_SALIDAS / 'informe_grafico_1.png', dpi=300, bbox_inches='tight')
plt.close(fig_informe)

In [ ]:
resumen_conciertos = (afectadas.assign(tipo=np.where(afectadas['seat_in_conflict'], 'con doble vigencia (localidad revendida)', 'sin segunda entrada'))
                      .groupby(['concert_id', 'tipo']).size().unstack(fill_value=0)
                      .reindex(sorted(df_concerts['concert_id']), fill_value=0)
                      .reindex(columns=['con doble vigencia (localidad revendida)', 'sin segunda entrada'], fill_value=0))


def grafico_por_concierto(figsize, base, compacto=False):
    '''Gráfico 2: entradas indebidas por concierto. `compacto` da la versión para el informe.'''
    fig, eje = plt.subplots(figsize=figsize)
    resumen_conciertos.plot.bar(stacked=True, ax=eje, color=[COLOR_INDEBIDO, '#e9a99d'], rot=0, width=0.65)
    eje.set_title('Entradas indebidas por concierto' if compacto else 'Entradas emitidas con la reserva caducada, por concierto', loc='left', fontsize=base + 1)
    eje.set_xlabel('')
    eje.set_ylabel('entradas', fontsize=base)
    eje.tick_params(labelsize=base)
    companias = df_concerts.set_index('concert_id')['ticketing_company_id']
    if compacto:
        # Etiqueta corta de dos líneas (nº de concierto y proveedor) para que no se solapen en un gráfico pequeño
        eje.set_xticklabels([f'{c[-2:]}\n{companias[c]}' for c in resumen_conciertos.index], fontsize=base)
        eje.set_xlabel('concierto (CON-) y proveedor', fontsize=base)
    else:
        for i, concierto in enumerate(resumen_conciertos.index):
            eje.annotate(companias[concierto], xy=(i, -0.16), xycoords=('data', 'axes fraction'), ha='center', fontsize=base, color='gray', annotation_clip=False)
    eje.legend(['Con doble vigencia', 'Sin segunda entrada'] if compacto else ['con doble vigencia (localidad revendida)', 'sin segunda entrada'], fontsize=base, frameon=False)
    fig.tight_layout()
    return fig


fig = grafico_por_concierto((8, 3.6), 8)
fig.savefig(CARPETA_SALIDAS / 'grafico_2_entradas_indebidas_por_concierto.png', dpi=200, bbox_inches='tight')
plt.show()

fig_informe = grafico_por_concierto((2.9, 2.7), 6.5, compacto=True)
fig_informe.savefig(CARPETA_SALIDAS / 'informe_grafico_2.png', dpi=300, bbox_inches='tight')
plt.close(fig_informe)

## 12. Validación

Que el código termine sin errores no demuestra que los recuentos sean correctos. Se escribe una función de validación con `assert` que comprueba que las cifras clave son **consistentes entre sí** por caminos distintos: la tabla maestra conserva una fila por reserva, los estados reparten todas las reservas sin solapes, los importes cuadran con la tabla de pagos, y las dos vías para contar los conflictos (por entradas y por reservas) coinciden.

In [ ]:
def validar_integracion():
    '''Se comprueba que las cifras clave son coherentes entre sí; cada assert detiene el proceso si falla.'''
    def comprobar(condicion, descripcion):
        assert condicion, f'FALLO: {descripcion}'
        print(f'  ok · {descripcion}')

    comprobar(len(df_maestra) == len(df_reservations) and df_maestra['reservation_id'].is_unique, 'la tabla maestra tiene una fila por reserva')
    comprobar(df_maestra['reservation_state'].value_counts().sum() == len(df_reservations), 'los estados reparten todas las reservas, sin solapes ni huecos')
    comprobar(int(df_maestra['has_payment'].sum()) == len(df_pagos), 'las reservas con pago coinciden con la tabla de pagos')
    comprobar(int(df_maestra['has_ticket'].sum()) == len(df_tickets), 'las reservas con entrada coinciden con la tabla de entradas')
    comprobar(np.isclose(df_maestra['capture_amount_eur'].sum(), df_pagos['capture_amount_eur'].sum()), 'el cobrado de la maestra coincide con el de los pagos')
    comprobar(np.isclose(df_maestra['refund_amount_eur'].sum(), df_pagos['refund_amount_eur'].sum()), 'el devuelto de la maestra coincide con el de los pagos')
    comprobar(int((df_maestra['reservation_state'] == 'SIN_PAGO_CADUCADA').sum()) == int(reservas_sin_pago.sum()), 'las reservas sin pago coinciden con el cruce de claves')
    comprobar(not df_maestra.loc[~df_maestra['has_payment'], 'has_ticket'].any(), 'ninguna entrada se emitió sin captura (P3)')
    comprobar(int(reservas_ordenadas['solapa'].sum()) == 0, 'ninguna reserva solapa con otra de su localidad')
    comprobar(afectadas['ticket_id'].is_unique and afectadas['reservation_id'].is_unique, 'cada reserva afectada tiene una sola entrada')
    comprobar(set(afectadas['ticket_id']) <= set(df_tickets['ticket_id']), 'las entradas indebidas existen en tickets')
    comprobar(int(entradas['solapa'].sum()) == len(localidades_conflicto), 'cada localidad en conflicto tiene exactamente una entrada que solapa')
    comprobar(int(entradas['en_conflicto'].sum()) == 2 * len(localidades_conflicto), 'cada conflicto implica exactamente dos entradas')
    comprobar(int((afectadas['seat_in_conflict']).sum()) == len(localidades_conflicto), 'cada conflicto tiene una entrada indebida y una contraparte legítima')
    comprobar(not afectadas['is_refunded'].any(), 'ninguna reserva con entrada indebida se ha devuelto')
    comprobar(len(contraparte) == len(localidades_conflicto), 'una contraparte por localidad en conflicto')


validar_integracion()
print('\nValidación superada.')

## 13. Exportación de evidencias

Se guardan en `outputs/` los dos gráficos del apartado 11 como PNG (versión normal y versión compacta `informe_grafico_*.png` para el informe) y las listas que permitirán a StageOps **reproducir una cifra o seguir un caso** durante la revisión: las reservas afectadas con todos sus identificadores y relojes, las localidades en conflicto, la clasificación de todas las reservas y los indicadores de alcance. Las fechas van en ISO 8601 UTC.

In [ ]:
def exportar(tabla, nombre):
    salida = tabla.copy()
    for columna in salida.select_dtypes(include='datetimetz').columns:
        salida[columna] = salida[columna].dt.strftime(FORMATO_FECHA)
    ruta = CARPETA_SALIDAS / nombre
    salida.to_csv(ruta, index=False, encoding='utf-8')
    relectura = pd.read_csv(ruta)
    assert relectura.shape == salida.shape, f'{nombre}: el fichero cambió al exportar'
    print(f'{nombre:42s} {relectura.shape[0]:>7,} filas · {relectura.shape[1]:>2} columnas')


columnas_afectadas = ['reservation_id', 'order_id', 'customer_id', 'concert_id', 'seat_id', 'ticketing_company_id', 'payment_id',
                      'created_at', 'expires_at', 'capture_occurred_at', 'capture_received_at', 'seconds_after_expiry',
                      'ticket_id', 'issued_at', 'issuer_version', 'capture_amount_eur', 'seat_in_conflict']
exportar(afectadas[columnas_afectadas].sort_values('issued_at'), 'reservas_entrada_indebida.csv')

conflictos = (entradas[entradas['en_conflicto']]
              .merge(df_maestra.loc[df_maestra['has_ticket'], ['ticket_id', 'order_id', 'customer_id', 'created_at', 'expires_at', 'capture_received_at', 'capture_amount_eur']],
                     on='ticket_id', how='left', validate='one_to_one')
              .assign(role=lambda t: np.where(t['ticket_id'].isin(tickets_indebidos), 'entrada indebida (confirmación tardía)', 'contraparte legítima'))
              [['concert_id', 'seat_id', 'role', 'ticket_id', 'reservation_id', 'order_id', 'customer_id', 'created_at', 'expires_at',
                'capture_received_at', 'issued_at', 'voided_at', 'issuer_version', 'capture_amount_eur']]
              .sort_values(['concert_id', 'seat_id', 'issued_at']))
exportar(conflictos, 'localidades_doble_vigencia.csv')

exportar(df_maestra.groupby(['reservation_state', 'ticketing_company_id']).agg(reservations=('reservation_id', 'size'), capture_amount_eur=('capture_amount_eur', 'sum'),
                                                                      refund_amount_eur=('refund_amount_eur', 'sum')).reset_index(), 'estados_de_reserva.csv')
exportar(indicadores, 'indicadores_alcance.csv')
exportar(df_secuencia, 'caso_secuencia.csv')

## 14. Conclusiones

### 14.1 Respuesta a las cinco preguntas de dirección

**1. ¿Qué conciertos, periodos y compañías presentan incidencias demostrables?**
Solo **EntradaLink**, en **CON-04** (desde el 16/10/2026 a las 10:20 UTC) y **CON-06** (23/10/2026). No hay ninguna incidencia de este tipo en LumaTicket ni en CON-02 (EntradaLink antes del cambio). Todas las entradas afectadas llevan la versión **4.8.0** del emisor.

**2. ¿Qué parece repetido pero es legítimo y qué es un conflicto real?**

| Parece repetido | Veredicto | Evidencia |
| :--- | :--- | :--- |
| Notificaciones de pago reenviadas | **Legítimo** | 4.463 reservas con reenvío; cada captura se cuenta una vez |
| Localidades reservadas varias veces | **Legítimo** | 9.307 localidades con más de una reserva y **0 solapes** de ventanas |
| Localidades revendidas tras anular la entrada | **Legítimo** | 3.565 reventas, siempre con la anterior anulada antes |
| **Dos entradas vigentes a la vez** | **Conflicto real** | **1.195 localidades**, solo en CON-04 y CON-06 |

**3. ¿Qué secuencia explica los casos afectados?**
La reserva caduca a los 10 minutos; la localidad queda libre y **otra persona la reserva, la paga y recibe su entrada**; después llega la confirmación del primer cobro, **fuera de plazo**, y el emisor genera también una entrada para la primera reserva. La regla del servicio ordena devolver el importe sin emitir. Ejemplo rastreable en el apartado 9: CON-04, localidad `S19-R18-B21`, reservas `RES-0108527` (confirmación 192 s tras caducar) y `RES-0108528`, entradas `TKT-0101389` y `TKT-0101390`.

**4. ¿Cuántas butacas, reservas, compradores e importes están afectados?** (definiciones y denominadores en el apartado 10)

| Indicador | Valor |
| :--- | ---: |
| Reservas con entrada emitida tras caducidad (entradas indebidas) | **2.188** (3,95 % de las reservas con pago de EntradaLink desde el cambio) |
| · de ellas, en localidades con doble vigencia | 1.195 |
| · de ellas, en localidades sin segunda entrada | 993 |
| Entradas implicadas en conflictos de butaca | 2.390 |
| Pedidos con alguna entrada indebida | 2.116 |
| Compradores con entrada indebida (`customer_id` distintos) | 2.086 |
| Compradores perjudicados (la otra entrada vigente) | 1.186 |
| Compradores afectados en total, sin duplicar | **3.244** (5,41 % de los compradores) |
| Cobrado en reservas con entrada indebida | **187.738 EUR** (103.431 con conflicto + 84.307 sin él) |

Cobrado total: 15.341.969 EUR. Devuelto: 435.070 EUR, de los que 126.277 EUR corresponden a confirmaciones tardías **bien gestionadas** y 308.793 EUR a anulaciones. Los 187.738 EUR son un importe que **podría requerir regularización**, no una pérdida: según la regla debió devolverse, y la decisión de cómo (devolver a quien llegó tarde, reubicar al perjudicado) es de StageOps.

**5. ¿Qué corregir antes de la siguiente apertura de ventas?** Ver 14.4.

### 14.2 Qué está demostrado

- Las seis fuentes se integran sin claves huérfanas ni uniones que multipliquen filas; la reserva es la unidad de enlace.
- **El bloqueo de diez minutos funciona**: ninguna reserva se solapa con otra de su localidad.
- La entrada se emite **exactamente 2 segundos después de la primera recepción** de la confirmación, en las 176.562 entradas. El reloj de la regla es la recepción, no el cobro.
- **Con la regla `CURRENT_OWNER_AND_NOT_EXPIRED` vigente, toda confirmación tardía se devuelve sin emitir** (1.481 casos, EntradaLink antes del cambio y LumaTicket siempre). **Con `PAYMENT_REFERENCE_EXISTS` vigente, toda confirmación tardía emite entrada** (2.188 casos) y ninguna se devuelve.
- Desde `CHG-004`, EntradaLink recibe **más confirmaciones tardías** que su línea de base: ≈4,0 % frente a ≈1,1-1,3 % en el resto de conciertos, incluido LumaTicket.
- Las confirmaciones tardías son una población aparte (retraso de 516 a 1.063 s; ninguna confirmación normal supera los 355 s).

### 14.3 Qué no está demostrado y qué información adicional hace falta

| Cuestión abierta | Por qué no se puede afirmar | Información necesaria |
| :--- | :--- | :--- |
| Que `CHG-004` **cause** las entradas indebidas | Hay una asociación fuerte (antes/después y frente a LumaTicket), pero no una prueba: no se conoce el comportamiento interno del emisor | Código o configuración del emisor 4.8.0; confirmación del proveedor de qué hace `PAYMENT_REFERENCE_EXISTS`; prueba con la regla anterior en un entorno de pruebas |
| **Por qué aumentan** las confirmaciones tardías en CON-04 y CON-06 | La descripción de `CHG-004` (*«Actualización del adaptador de confirmación de pagos»*) lo hace compatible, pero también podría ser carga en esas aperturas | Registros de latencia de la pasarela y del adaptador; comparación con otras aperturas |
| Cómo regularizar cada caso | Depende de la política comercial (devolver o reubicar) | Decisión de StageOps y contacto con los compradores |
| Butacas nunca solicitadas | El paquete no incluye el inventario completo | No aplica: fuera del alcance de las exportaciones |
| Acceso al recinto | Los conciertos no se han celebrado: no hay escaneos | No aplica: el alcance es venta y asignación |

### 14.4 Tres medidas priorizadas y cómo comprobar que funcionan

**Medida 1 · Antes de la próxima apertura: impedir que se emita una entrada con la reserva caducada** (restablecer en el emisor de EntradaLink la validación de vigencia y propiedad, o revertir `CHG-004`; como mínimo, bloquear la emisión cuando la recepción sea posterior a `expires_at`).
*Cómo se verifica:* (a) en un entorno de pruebas, una confirmación con retraso superior a 600 s debe terminar en **devolución sin entrada**; (b) tras la próxima apertura, ejecutar este notebook sobre la nueva exportación: debe haber **0** reservas en `TARDIA_CON_ENTRADA` y **0** localidades con dos entradas vigentes, con confirmaciones tardías presentes (≈1-4 % de las reservas) y **todas devueltas**.

**Medida 2 · Regularizar los casos ya existentes** (`outputs/reservas_entrada_indebida.csv` y `outputs/localidades_doble_vigencia.csv` con todos los identificadores). Primero las **1.195 localidades con doble vigencia**: la contraparte compró dentro de las reglas, por lo que debe conservar la localidad; la entrada tardía se anula y se devuelve (103.431 EUR). Después las **993 entradas sin conflicto de butaca** (84.307 EUR), que requieren una decisión comercial.
*Cómo se verifica:* tras las correcciones, **0 localidades con dos entradas vigentes** y una conciliación por reserva en la que toda captura queda en exactamente uno de estos estados: entrada vigente, o devolución sin entrada; el neto cobrado debe igualar el importe de las entradas vigentes.

**Medida 3 · Detectar y prevenir en continuo.** (a) Un control diario con tres alertas: entrada emitida con `issued_at` ≥ `expires_at`, localidad con dos entradas vigentes, y captura sin entrada ni devolución; (b) tratar todo cambio de configuración del emisor como un cambio con prueba de regresión y despliegue canario contra una compañía de referencia; (c) pedir al proveedor el análisis del aumento de latencia de confirmación.
*Cómo se verifica:* las tres alertas valen 0 durante las siguientes aperturas; la frecuencia de confirmaciones tardías de EntradaLink vuelve a la línea de base (≈1-1,3 %); y cada cambio futuro de `ticket_issuer` se acompaña de una comparación antes/después como la del apartado 8.2.

### 14.5 Límites de estas conclusiones

- Los números se calculan sobre las exportaciones hasta el cierre de datos (2026-10-28 23:59:59 UTC); lo que ocurra después no está cubierto.
- `customer_id` identifica al comprador, no a todos los asistentes: los compradores afectados son compradores, no personas.
- La asociación entre `CHG-004` y las incidencias es una hipótesis fuerte y contrastada con un grupo de comparación, **no una causalidad probada**.